
# Trees and Ensembles, Assignment
## A German bank and a Taiwanese bank, from entropy to gradient boosting

**Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

---

### What this is about

This assignment covers the four notebooks of the set on two new problems:

- **Part A.** German Credit (Hofmann, 1994): 1,000 consumer loans from a German bank,
  with categorical variables (checking account status, history, purpose) and numeric ones
  (amount, term, age). Here you compute entropy and gain by hand, build a tree,
  prune it and interpret it as a rulebook.
- **Part B.** Taiwan credit cards (Yeh and Lien, 2009): 30,000 customers. Here you
  compare trees and ensembles, choose hyperparameters with validation and discuss what is
  gained and what is lost.

**Python is not graded.** All the code is already written; what is graded is the reading of the
results and the decisions. Answer in the *Your answer* cells, in your own words and with the
numbers that come out of your own run.

**Submission format.** The executed notebook (with outputs visible) exported to PDF or
HTML, plus the `.ipynb` file.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import warnings, time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 30)
print("Ready.")



---
# Part A. German Credit: entropy, tree and pruning

## A.1 The data

1,000 loans; `class` is `bad` if the customer did not pay (30%) and `good` if they paid. We use seven
categorical columns and three numeric ones.

| Column | What it is |
|---|---|
| `checking_status` | Checking account status: no account, negative balance, 0 to 200 marks, more than 200 |
| `credit_history` | History: no credits, all paid, paying on time, delays, critical account |
| `purpose` | What the loan is for: new car, used car, furniture, TV, appliances, repairs, education, business, others |
| `savings_status` | Savings: less than 100, 100 to 500, 500 to 1,000, more than 1,000, no known savings |
| `employment` | Job tenure: unemployed, less than 1 year, 1 to 4, 4 to 7, more than 7 |
| `housing` | Housing: rents, owns, for free |
| `foreign_worker` | Foreign worker: yes / no |
| `duration` | Loan term, in months |
| `credit_amount` | Loan amount, in marks |
| `age` | Age, in years |
| `class` | **What we want to explain:** `bad` if they did not pay, `good` if they paid |


In [ ]:

german = fetch_openml("credit-g", version=1, as_frame=True).frame
CATEG = ["checking_status", "credit_history", "purpose", "savings_status", "employment", "housing", "foreign_worker"]
NUMER = ["duration", "credit_amount", "age"]
german = german[CATEG + NUMER + ["class"]].copy()
for c in CATEG + ["class"]:
    german[c] = german[c].astype(str)
print("Customers:", len(german), "  Fraction that did not pay:", round((german["class"] == "bad").mean(), 3))
german.head(8)



## A.2 Entropy and gain, by hand

The same functions from Notebook 1, applied to the seven categorical columns.


In [ ]:

def entropia(serie):
    p = serie.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def ganancia(df, objetivo, atributo):
    h_cond = sum(len(g) / len(df) * entropia(g[objetivo]) for _, g in df.groupby(atributo))
    return entropia(df[objetivo]) - h_cond

print("H(class) =", round(entropia(german["class"]), 3), "bits")
tabla_ganancia = pd.DataFrame({
    "distinct values": [german[a].nunique() for a in CATEG],
    "H(attribute)": [entropia(german[a]) for a in CATEG],
    "gain": [ganancia(german, "class", a) for a in CATEG],
}, index=CATEG)
tabla_ganancia["gain ratio"] = tabla_ganancia["gain"] / tabla_ganancia["H(attribute)"]
tabla_ganancia.sort_values("gain", ascending=False).round(4)


In [ ]:

# The fraction that does not pay within each value of the attribute with the most gain
atributo_top = tabla_ganancia["gain"].idxmax()
resumen = german.groupby(atributo_top)["class"].agg(clientes="size", no_paga=lambda s: (s == "bad").mean()).sort_values("no_paga")
resumen.round(3)



**Question 1.** (a) $H(\text{class})$ is less than 1 bit: explain why, comparing it with the
14-firm table of the course, where it was 0.985. (b) Which attribute has the most gain and
how much uncertainty does it remove, in bits and as a percentage of $H(\text{class})$? (c) `purpose` has
10 distinct values: compare its position in the ranking by gain with its position by gain
ratio and explain the difference. (d) Using the table from the second cell, write in two lines the
business rule a credit analyst would draw from the winning attribute.



*Your answer:*



## A.3 A tree, and its honest grade

The categorical columns are expanded into columns of zeros and ones (`get_dummies`) so the
tree can split on them. We set aside 300 test customers **before** looking at anything.


In [ ]:

X_g = pd.get_dummies(german[CATEG + NUMER], columns=CATEG, drop_first=False).astype(float)
y_g = (german["class"] == "bad").astype(int)
Xg_train, Xg_test, yg_train, yg_test = train_test_split(X_g, y_g, test_size=300, random_state=1, stratify=y_g)
print("Training:", len(Xg_train), "  Test:", len(Xg_test), "  Columns:", X_g.shape[1])

filas = []
for d in [1, 2, 3, 4, 5, 6, 8, 10, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xg_train, yg_train)
    filas.append((str(d), m.get_n_leaves(), accuracy_score(yg_train, m.predict(Xg_train)), accuracy_score(yg_test, m.predict(Xg_test)),
                  roc_auc_score(yg_test, m.predict_proba(Xg_test)[:, 1])))
curva = pd.DataFrame(filas, columns=["depth", "leaves", "training accuracy", "test accuracy", "test AUC"])
print(f"Always predicting 'pays' gets {1 - yg_test.mean():.3f} right on test")
curva.round(3)


In [ ]:

arbol3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xg_train, yg_train)
print(export_text(arbol3, feature_names=list(X_g.columns), show_weights=True))
cm = confusion_matrix(yg_test, arbol3.predict(Xg_test))
print(pd.DataFrame(cm, index=["actual pays", "actual does not pay"], columns=["predicted pays", "predicted does not pay"]))



**Question 2.** (a) At what depth does the test accuracy stop improving? And the AUC?
Do they agree? (b) The unrestricted tree gets 100% on training: how many leaves does it have and what
accuracy does it give on test? Explain in your own words what it learned in excess. (c) Read the
depth-3 tree and write it as three business rules ("if ... and ..., then high risk").
Does the attribute that won in Question 1 appear at the root? (d) Look at the confusion matrix: of
the customers who did not pay, what fraction does the tree detect? Is a model that gets 72% right
but detects few bad customers useful to a bank? What would you prefer to change?



*Your answer:*



## A.4 Pruning without looking at the test set

We choose the complexity with 5-fold cross-validation on the 700 training customers,
searching over `ccp_alpha` and `min_samples_leaf`. The test set is used only once, at the end.


In [ ]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
alfas = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xg_train, yg_train).ccp_alphas
alfas = np.unique(np.round(alfas[alfas < 0.05], 4))
busqueda = GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alfas, "min_samples_leaf": [1, 5, 10, 20, 40]},
                        cv=cv, scoring="roc_auc").fit(Xg_train, yg_train)
podado = busqueda.best_estimator_
print("Chosen by cross-validation:", busqueda.best_params_, f"  validation AUC {busqueda.best_score_:.3f}")
print(f"Leaves: {podado.get_n_leaves()}   test accuracy {accuracy_score(yg_test, podado.predict(Xg_test)):.3f}   test AUC {roc_auc_score(yg_test, podado.predict_proba(Xg_test)[:, 1]):.3f}")
print()
print(export_text(podado, feature_names=list(X_g.columns)))



**Question 3.** (a) How many leaves does the tree chosen by cross-validation have and how does
its test AUC compare with the best one in the table of Question 2? (b) In Question 2
we chose the depth by looking at the test set; here we did not. Explain why the grade of Question 2
is "inflated" even though it is higher, and which of the two you would report to your boss. (c) Propose a
pre-pruning rule that makes business sense (for example, a minimum number of customers per leaf) and
justify the number.



*Your answer:*



---
# Part B. Taiwan: trees against ensembles

## B.1 The data

30,000 credit card customers; `target` is 1 if the customer did not pay the following month
(22%). The variables: credit limit, sex, education, marital status, age, and for each
of the last six months the payment delay, the bill amount and the amount paid.
We hold out 9,000 test customers. Since only 22% do not pay, the metric is the **AUC**.


In [ ]:

taiwan = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
X_t = taiwan.data.apply(pd.to_numeric); y_t = taiwan.target.astype(int)
meses = ["sep", "ago", "jul", "jun", "may", "abr"]
X_t.columns = (["limite_credito", "sexo", "educacion", "estado_civil", "edad"] + ["atraso_" + m for m in meses]
               + ["factura_" + m for m in meses] + ["pago_" + m for m in meses])
Xt_train, Xt_test, yt_train, yt_test = train_test_split(X_t, y_t, test_size=9000, random_state=0, stratify=y_t)
print("Training:", len(Xt_train), "  Test:", len(Xt_test), "  Fraction that did not pay:", round(y_t.mean(), 3))



## B.2 Five models, one table


In [ ]:

# --- PART B PARAMETERS (Question 5 changes them) ---
N_ARBOLES = 200          # trees of the random forest and stumps of the AdaBoost
RONDAS = 300             # rounds of the gradient boosting
TASA = 0.05              # learning rate of the gradient boosting
PROF_GB = 4              # depth of each gradient boosting tree

def correr_modelos(Xtr, ytr, Xte, yte):
    modelos = {
        "depth-3 tree":                          DecisionTreeClassifier(max_depth=3, random_state=0),
        "unrestricted tree":                     DecisionTreeClassifier(random_state=0),
        f"random forest, {N_ARBOLES} trees":     RandomForestClassifier(n_estimators=N_ARBOLES, min_samples_leaf=5, n_jobs=-1, random_state=0),
        f"AdaBoost, {N_ARBOLES} stumps":         AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=N_ARBOLES, random_state=0),
        f"gradient boosting, {RONDAS} rounds":   HistGradientBoostingClassifier(max_iter=RONDAS, learning_rate=TASA, max_depth=PROF_GB, random_state=0),
    }
    filas = []
    for nombre, m in modelos.items():
        t0 = time.time(); m.fit(Xtr, ytr); seg = time.time() - t0
        filas.append((nombre, accuracy_score(ytr, m.predict(Xtr)), accuracy_score(yte, m.predict(Xte)), roc_auc_score(yte, m.predict_proba(Xte)[:, 1]), seg))
    return pd.DataFrame(filas, columns=["model", "training accuracy", "test accuracy", "test AUC", "seconds"]).set_index("model"), modelos

resultados, modelos = correr_modelos(Xt_train, yt_train, Xt_test, yt_test)
print(f"Always predicting 'pays' gets {1 - yt_test.mean():.3f} right on test")
resultados.round(3)



**Question 4.** (a) Why is the test accuracy almost the same for all models and
not useful for comparing them here? (b) Rank the models by test AUC. How much does the best
ensemble gain over the depth-3 tree, and over the unrestricted tree? (c) The unrestricted tree has the
best training accuracy and the worst test AUC: explain it with the vocabulary of
Notebook 3. (d) With the 219 firms of the course the ensembles did not beat the small tree;
here they do. What changed?



*Your answer:*



## B.3 Experiment: less data

Go back to B.2 and run the table training with only the first 1,000 training
customers (the cell below does it). Then with 5,000.


In [ ]:

for n_ in [1000, 5000]:
    res_n, _ = correr_modelos(Xt_train.iloc[:n_], yt_train.iloc[:n_], Xt_test, yt_test)
    print(f"--- training with {n_} customers ---")
    print(res_n[["test accuracy", "test AUC"]].round(3))
    print()



**Question 5.** (a) With 1,000 customers, what is the AUC difference between the best ensemble
and the depth-3 tree? And with 5,000 and with 21,000? (b) Which model benefits most from
having more data, and which barely changes? (c) Go back to B.2 and try `RONDAS = 1000` with
`TASA = 0.05`, and then `RONDAS = 300` with `TASA = 0.5`. Write down the AUCs. What would you tell someone
who proposes "more rounds and faster"?



*Your answer:*



## B.4 What the model looks at, and how a decision is explained


In [ ]:

rf = modelos[f"random forest, {N_ARBOLES} trees"]
importancia = pd.Series(rf.feature_importances_, index=X_t.columns).sort_values(ascending=False)
fig = px.bar(importancia.head(12), orientation="h", title="Variable importance of the random forest (Taiwan)", labels={"value": "importance", "index": ""})
fig.show()

# The fraction that does not pay by September delay (the variable that usually wins)
tabla_atraso = pd.DataFrame({"customers": Xt_train.groupby(Xt_train["atraso_sep"]).size(),
                             "fraction that does not pay": yt_train.groupby(Xt_train["atraso_sep"]).mean()})
tabla_atraso.round(3)



**Question 6.** (a) Which variables weigh most in the random forest? Does it make business sense?
(b) Look at the September delay table: is the relationship with not paying a straight line, a step
or something else? How would a logistic regression on `atraso_sep` see it and how does a
tree see it? Connect with the U-shape of Notebook 1. (c) A customer asks for an explanation of why the
card was denied. With the depth-3 tree of Part A the explanation is one sentence; with the random
forest, what would you show them? (d) To close, in four or five lines: if you were the risk
manager, would you use the gradient boosting (better AUC) or the depth-3 tree (explainable)?
What conditions would you set in each case?



*Your answer:*
